In [12]:
import pandas as pd
import requests

# URL da API para os dados KOI
API_URL_KOI = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query=select+*+from+cumulative&format=json"

print("🚀 Buscando dados da API da NASA (Kepler Objects of Interest)...")

try:
    # Faz a requisição para a API
    response_koi = requests.get(API_URL_KOI)
    response_koi.raise_for_status() # Verifica se a requisição foi bem-sucedida

    # Converte a resposta JSON em um DataFrame
    print("✅ Dados KOI recebidos! Convertendo para DataFrame...")
    data_koi = response_koi.json()
    df_koi = pd.DataFrame(data_koi)

    print(f"🎉 Sucesso! DataFrame 'df_koi' criado com {df_koi.shape[0]} linhas.")
    
except requests.exceptions.RequestException as e:
    print(f"❌ Erro ao buscar dados KOI: {e}")

# Verificando as primeiras linhas (opcional)
# print(df_koi.head())

🚀 Buscando dados da API da NASA (Kepler Objects of Interest)...
✅ Dados KOI recebidos! Convertendo para DataFrame...
🎉 Sucesso! DataFrame 'df_koi' criado com 9564 linhas.


Carrega dados do TOI

In [9]:
import pandas as pd
import requests

# URL da API para os dados TOI
API_URL_TOI = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query=select+*+from+toi&format=json"

print("🚀 Buscando dados da API da NASA (TESS Objects of Interest)...")

try:
    # Faz a requisição para a API
    response_toi = requests.get(API_URL_TOI)
    response_toi.raise_for_status() # Verifica se a requisição foi bem-sucedida

    # Converte a resposta JSON em um DataFrame
    print("✅ Dados TOI recebidos! Convertendo para DataFrame...")
    data_toi = response_toi.json()
    df_toi = pd.DataFrame(data_toi)

    print(f"🎉 Sucesso! DataFrame 'df_toi' criado com {df_toi.shape[0]} linhas.")

except requests.exceptions.RequestException as e:
    print(f"❌ Erro ao buscar dados TOI: {e}")

# Verificando as primeiras linhas (opcional)
# print(df_toi.head())

🚀 Buscando dados da API da NASA (TESS Objects of Interest)...
✅ Dados TOI recebidos! Convertendo para DataFrame...
🎉 Sucesso! DataFrame 'df_toi' criado com 7703 linhas.


Unificando os Dados KOI e TOI para o Modelo

In [13]:
# --- Passo 1: Mapear e Padronizar o DataFrame KOI (Kepler) ---
print("⚙️  Processando dados do Kepler (KOI)...")

# Dicionário de mapeamento: {'nome_original': 'nome_novo_padronizado'}
map_koi = {
    'koi_disposition': 'target_disposition',
    'koi_period': 'orbital_period',
    'koi_duration': 'transit_duration_hr',
    'koi_depth': 'transit_depth_ppm',
    'koi_prad': 'planet_radius_earth',
    'koi_steff': 'stellar_temp_k',
    'koi_srad': 'stellar_radius_solar'
}

# Seleciona apenas as colunas do dicionário e as renomeia
df_koi_std = df_koi[list(map_koi.keys())].rename(columns=map_koi)

# Mapeia os valores do alvo para 0 (Falso Positivo) e 1 (Planeta)
target_map_koi = {'CONFIRMED': 1, 'CANDIDATE': 1, 'FALSE POSITIVE': 0}
df_koi_std['target'] = df_koi_std['target_disposition'].map(target_map_koi)

# Adiciona uma coluna para identificar a origem dos dados
df_koi_std['source'] = 'Kepler'


# --- Passo 2: Mapear e Padronizar o DataFrame TOI (TESS) ---
print("⚙️  Processando dados do TESS (TOI)...")

# Dicionário de mapeamento para os dados do TESS
map_toi = {
    'tfopwg_disp': 'target_disposition',
    'pl_orbper': 'orbital_period',
    'pl_trandurh': 'transit_duration_hr',
    'pl_trandep': 'transit_depth_ppm',
    'pl_rade': 'planet_radius_earth',
    'st_teff': 'stellar_temp_k',
    'st_rad': 'stellar_radius_solar'
}

df_toi_std = df_toi[list(map_toi.keys())].rename(columns=map_toi)

# Mapeia os valores do alvo (KP = Known Planet, CP = Candidate Planet, FP = False Positive)
target_map_toi = {'KP': 1, 'CP': 1, 'FP': 0}
df_toi_std['target'] = df_toi_std['target_disposition'].map(target_map_toi)

# Adiciona a coluna de origem
df_toi_std['source'] = 'TESS'


# --- Passo 3: Combinar os DataFrames e Limpeza Final ---
print("🤝 Combinando os dois DataFrames...")

# Usa pd.concat para empilhar um sobre o outro
df_combined = pd.concat([df_koi_std, df_toi_std], ignore_index=True)

# Limpeza final:
# 1. Remove linhas onde o alvo não pôde ser mapeado (virou NaN)
df_combined.dropna(subset=['target'], inplace=True)
# 2. Converte a coluna 'target' para inteiro
df_combined['target'] = df_combined['target'].astype(int)
# 3. Remove a coluna de texto original do alvo, que não será mais usada
df_combined.drop(columns=['target_disposition'], inplace=True)


# --- Verificação Final ---
print("\n🎉 DataFrame combinado criado com sucesso!")
print(f"Total de amostras: {df_combined.shape[0]}")
print("\nDistribuição da fonte de dados:")
print(df_combined['source'].value_counts())
print("\nDistribuição do alvo (0 = Falso Positivo, 1 = Planeta):")
print(df_combined['target'].value_counts())
print("\nVerificando as primeiras 5 linhas do DataFrame final:")
print(df_combined.head())
print("\nVerificando as últimas 5 linhas do DataFrame final:")
print(df_combined.tail())

⚙️  Processando dados do Kepler (KOI)...
⚙️  Processando dados do TESS (TOI)...
🤝 Combinando os dois DataFrames...

🎉 DataFrame combinado criado com sucesso!
Total de amostras: 12028

Distribuição da fonte de dados:
source
Kepler    9564
TESS      2464
Name: count, dtype: int64

Distribuição do alvo (0 = Falso Positivo, 1 = Planeta):
target
0    6036
1    5992
Name: count, dtype: int64

Verificando as primeiras 5 linhas do DataFrame final:
   orbital_period  transit_duration_hr  transit_depth_ppm  \
0        9.488036              2.95750              615.8   
1       54.418383              4.50700              874.8   
2       19.899140              1.78220            10829.0   
3        1.736952              2.40641             8079.2   
4        2.525592              1.65450              603.3   

   planet_radius_earth  stellar_temp_k  stellar_radius_solar  target  source  
0                 2.26          5455.0                 0.927       1  Kepler  
1                 2.83         

Pré-processamento e Seleção de features

In [4]:
# Criar uma cópia para evitar warnings
model_df = df.copy()

# Mapear o alvo para valores numéricos: 1 para planetas (confirmados ou candidatos) e 0 para falsos positivos.
target_map = {'CONFIRMED': 1, 'CANDIDATE': 1, 'FALSE POSITIVE': 0}
model_df['target'] = model_df['koi_disposition'].map(target_map)

# Verificar a distribuição do novo alvo
print("Distribuição da variável alvo:")
print(model_df['target'].value_counts())

Distribuição da variável alvo:
target
0    4839
1    4725
Name: count, dtype: int64


In [6]:
# Lista de colunas que usaremos como features
feature_cols = [
    # Features de Trânsito Primárias (as mais importantes)
    'koi_period',        # Período orbital
    'koi_duration',      # Duração do trânsito
    'koi_depth',         # Profundidade do trânsito
    'koi_model_snr',     # Relação sinal-ruído (MUITO PODEROSA)

    # Features Planetárias e Orbitais
    'koi_prad',          # Raio do planeta
    'koi_teq',           # Temperatura de equilíbrio
    'koi_impact',        # Parâmetro de impacto (quão central é o trânsito)
    'koi_insol',         # Fluxo de insolação

    # Flags de Falso Positivo (MINA DE OURO)
    'koi_fpflag_nt',     # Flag: Não parece um trânsito
    'koi_fpflag_ss',     # Flag: Variabilidade estelar
    'koi_fpflag_co',     # Flag: Deslocamento do centroide
    'koi_fpflag_ec',     # Flag: Binária eclipsante
]

# Separar features (X) e alvo (y)
X = model_df[feature_cols]
y = model_df['target']

# Tratar valores ausentes (NaN). Uma estratégia simples e eficaz é preencher com 0.
X = X.fillna(0)

print("\nFeatures selecionadas para o modelo:")
print(X.head())


Features selecionadas para o modelo:
   koi_period  koi_duration  koi_depth  koi_model_snr  koi_prad  koi_teq  \
0    9.488036       2.95750      615.8           35.8      2.26    793.0   
1   54.418383       4.50700      874.8           25.8      2.83    443.0   
2   19.899140       1.78220    10829.0           76.3     14.60    638.0   
3    1.736952       2.40641     8079.2          505.6     33.46   1395.0   
4    2.525592       1.65450      603.3           40.9      2.75   1406.0   

   koi_impact  koi_insol  koi_fpflag_nt  koi_fpflag_ss  koi_fpflag_co  \
0       0.146      93.59              0              0              0   
1       0.586       9.11              0              0              0   
2       0.969      39.30              0              0              0   
3       1.276     891.96              0              1              0   
4       0.701     926.16              0              0              0   

   koi_fpflag_ec  
0              0  
1              0  
2        

Treinamento do modelo XGBoost

In [7]:
from sklearn.model_selection import train_test_split
import xgboost as xgb

# Dividir os dados em conjuntos de treino e teste
# stratify=y é importante para manter a proporção de planetas/não-planetas nos dois conjuntos
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.3,    # 30% dos dados para teste
    random_state=42,  # Para reprodutibilidade
    stratify=y
)

print(f"Tamanho do conjunto de treino: {X_train.shape[0]} amostras")
print(f"Tamanho do conjunto de teste: {X_test.shape[0]} amostras")

# Instanciar e treinar o modelo XGBoost
# Dica de Copilot: Digite '# train an xgboost classifier' e veja a mágica
model = xgb.XGBClassifier(
    objective='binary:logistic', 
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42
)

print("\n🚀 Treinando o modelo XGBoost...")
model.fit(X_train, y_train)
print("✅ Modelo treinado com sucesso!")

ModuleNotFoundError: No module named 'sklearn'